# Musanga V4 — NB02: V3_1-proven RGB U-Net with cached chips, 78-plot spatial CV

**Implementation follows the working V3_1/02B recipe**: pretrained ResNet18 U-Net from `segmentation_models_pytorch`, one-time 256 px chip preparation from a persistent raster handle per mosaic, compressed resumable `.npz` caches, GPU mixed precision, and an in-memory training dataset. Crucially, **there are no repeated Google Drive raster reads inside the training epochs**.

V4 differences are **only** the 78 exhaustive field plots, their crown GPKG, six RGB mosaics, joint plot–circle folds, and the final deduplicated window manifest. Never mix legacy V3_1 checkpoints with this reference.

Start with `SMOKE_TEST=True` (small cache, one fold). Run full 3-fold spatial validation separately.


In [ ]:
# 00 — Environment, frozen inputs and run settings
%pip -q install geopandas pyogrio shapely rasterio scipy scikit-learn torch matplotlib segmentation-models-pytorch
from pathlib import Path
import json, math, random, copy, time, warnings, os
import numpy as np, pandas as pd, geopandas as gpd, rasterio
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from rasterio.windows import Window
from rasterio.features import rasterize
from shapely.geometry import box
from shapely.ops import unary_union
from shapely import make_valid
from shapely.strtree import STRtree
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
DRIVE=Path('/content/drive/MyDrive')
ROOT=DRIVE/'Colab Notebooks/Musanga/Final'
assert ROOT.is_dir(), ROOT
DATA=ROOT/'Data'
REF_DIR=ROOT/'Outputs/v4/01_reference_field78_combined'
CONTRACT=json.loads((REF_DIR/'01_field78_reference_contract.json').read_text())
assert CONTRACT['schema']=='musanga_v4_field78_combined_reference_final_2',CONTRACT['schema']
NB00=json.loads(Path(CONTRACT['input_manifest']).read_text())
assert NB00['n_field_plots']==78
SPLITS=pd.read_csv(CONTRACT['cv_manifest'])
assert len(SPLITS)==9096,(len(SPLITS),'Expected 3032 unique windows × 3 folds; changed source audit?')
K=['cv_fold','mosaic','row','col','width','height']
assert not SPLITS.duplicated(K).any()
assert SPLITS.groupby('cv_fold').size().eq(3032).all()
assert set(SPLITS.split).issubset({'train','val','excluded_spatial_guard','excluded_no_validation_pixels'})
PLOTS=gpd.read_file(NB00['field_support_qa']).to_crs('EPSG:32635')
PLOT_FOLDS=pd.read_csv(CONTRACT['folds_csv'])
assert len(PLOTS)==len(PLOT_FOLDS)==78 and PLOT_FOLDS.source_id.is_unique
PLOTS=PLOTS.merge(PLOT_FOLDS[['source_id','site_key','cv_fold']],left_on='Plot_name',right_on='source_id',validate='one_to_one')
CIRCLE_FOLDS=pd.read_csv(REF_DIR/'01_field78_circle_folds.csv')
# Adjustable, deterministic experiment settings
SMOKE_TEST=True            # FIRST RUN; switch False for real training
RUN_FOLDS=[0]              # change to [0,1,2] for final evaluation
SEED=20261113
CHIP=256                   # 256 native pixels; no spatial downsampling
BATCH_SIZE=8
EPOCHS=1 if SMOKE_TEST else 25
STEPS_PER_EPOCH=3 if SMOKE_TEST else 100
LEARNING_RATE=2e-4
PATIENCE=4
POSITIVE_SAMPLE_BOOST=2.5
MAX_VAL_WINDOWS_PER_SITE=2 if SMOKE_TEST else 35
MAX_TRAIN_WINDOWS_PER_SITE=8 if SMOKE_TEST else 180
NUM_WORKERS=0              # Google Drive + GDAL: safest in Colab
OUT=ROOT/'Outputs/v4/02_rgb_unet_field78_v3recipe'/('smoke' if SMOKE_TEST else 'full_spatial_cv')
OUT.mkdir(parents=True,exist_ok=True)
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type=='cuda', 'V3_1 used GPU; select Colab GPU runtime before training'
print('Device:',DEVICE,'| train mode:', 'SMOKE TEST' if SMOKE_TEST else 'FULL SPATIAL CV')
print('Final manifest:',CONTRACT['cv_manifest'])
print('Observed window splits:');display(SPLITS.groupby(['cv_fold','site','split']).size().unstack(fill_value=0))


## 01 — Load and repair vector supervision once; verify exhaustive supports

The 78 plots and 34 circles are separate exhaustive areas. Any circle area inside a field plot is overridden by its field-plot annotation. Outside exhaustive areas, partial-site polygons label **their own pixels only**. Geometries are repaired conservatively (≤5% area shift), preserving source files. An explicit negative pixel is not inferred outside plots/circles.


In [ ]:
# 01 — Geometry references and fast spatial indexes
CROWNS=gpd.read_file(NB00['exhaustive_plot_crowns']).to_crs('EPSG:32635')
CIRCLE_ROI=gpd.read_file(NB00['circle_complete_roi']).to_crs('EPSG:32635')
CIRCLE_POS=gpd.read_file(NB00['circle_positive_crowns']).to_crs('EPSG:32635')
assert len(CROWNS)==3324 and len(CIRCLE_ROI)==34
assert 'Class_ID' in CROWNS
def polygons(g):
    if g is None or g.is_empty:return []
    if g.geom_type=='Polygon':return [g]
    if g.geom_type=='MultiPolygon':return list(g.geoms)
    return [a for v in getattr(g,'geoms',[]) for a in polygons(v)]
def fix(g):
    if g is None or g.is_empty:return None
    if g.is_valid:return g
    q=unary_union(polygons(make_valid(g)))
    if q.is_empty or not q.is_valid:return None
    if abs(q.area-g.area)/max(q.area,g.area,1e-12)>.05:return None
    return q
CROWNS.geometry=[fix(g) for g in CROWNS.geometry]
assert CROWNS.geometry.notna().all(),'Unrepaired plot crowns; see NB00 QA'
CIRCLE_ROI.geometry=[fix(g) for g in CIRCLE_ROI.geometry]
CIRCLE_POS.geometry=[fix(g) for g in CIRCLE_POS.geometry]
assert CIRCLE_ROI.geometry.notna().all()
CIRCLE_ROI['circle_id']=np.arange(len(CIRCLE_ROI))
CIRCLE_ROI=CIRCLE_ROI.merge(CIRCLE_FOLDS[['circle_id','site_key','cv_fold',
    'cross_fold_plot_intersection']],on='circle_id',validate='one_to_one')
CROWN_POS=list(CROWNS.loc[pd.to_numeric(CROWNS.Class_ID).eq(1)].geometry)
CROWN_NEG=list(CROWNS.loc[~pd.to_numeric(CROWNS.Class_ID).eq(1)].geometry)
CIRCLE_POS_GEOMS=[g for g in CIRCLE_POS.geometry if g is not None and not g.is_empty]
# Partial site references from old V3_1 audit, same positive=1 / other known class negative.
PARTIAL={}
for site,path in NB00['partial_site_crowns'].items():
    d=gpd.read_file(path).to_crs('EPSG:32635')
    d.geometry=[fix(g) for g in d.geometry]
    d=d[d.geometry.notna() & pd.to_numeric(d.Class_ID,errors='coerce').notna()]
    PARTIAL[site]=d
P_GEOMS=list(PLOTS.geometry)
C_GEOMS=list(CIRCLE_ROI.geometry)
P_T=STRtree(P_GEOMS); C_T=STRtree(C_GEOMS)
CP_T=STRtree(CROWN_POS);CN_T=STRtree(CROWN_NEG)
KPOS_T=STRtree(CIRCLE_POS_GEOMS)
PARTIAL_T={}
for site,d in PARTIAL.items():
    pos=list(d.loc[pd.to_numeric(d.Class_ID).eq(1)].geometry)
    neg=list(d.loc[~pd.to_numeric(d.Class_ID).eq(1)].geometry)
    PARTIAL_T[site]=(pos,neg,STRtree(pos) if pos else None,
                     STRtree(neg) if neg else None)
RGB={}
for item in sorted(SPLITS.mosaic.unique()):
    q=DATA/'Ortho_RGB'/item
    assert q.exists(),q
    with rasterio.open(q) as ds:
        assert ds.crs.to_epsg()==32635,(item,ds.crs)
        assert ds.count>=3,(item,ds.count)
        RGB[item]=dict(path=str(q),transform=ds.transform,site=next(
            iter(SPLITS.loc[SPLITS.mosaic==item,'site'].unique())))
# NOTE: ROI.shp is the authoritative six-mosaic modelling ROI (not the old
# circle-only support). The validation/labelled field plot is also required.
ROIS=gpd.read_file(DATA/'Ortho_RGB/ROI.shp').to_crs('EPSG:32635')
ROI_GEOMS=list(ROIS.geometry)
ROI_T=STRtree(ROI_GEOMS)
print('Loaded',len(PLOTS),'plots,',len(CIRCLE_ROI),'circles,',
      len(CROWN_POS),'Musanga plot crowns and',len(CIRCLE_POS_GEOMS),'circle crowns')


In [ ]:
# 02 — Rasterize strict plot > circle > partial supervision on demand
# Geometry-level spatial queries restrict operations to the local 256px footprint.
def geom_hits(tree,geoms,ext):
    if tree is None:return []
    return [geoms[int(i)] for i in tree.query(ext,predicate='intersects')]
_ACTIVE_RASTER=None
_ACTIVE_NAME=None
def rgb_window(raster_path,win):
    from contextlib import nullcontext
    name=Path(raster_path).name
    ds_context=nullcontext(_ACTIVE_RASTER) if (_ACTIVE_RASTER is not None and _ACTIVE_NAME==name) else rasterio.open(raster_path)
    with ds_context as ds:
        a=ds.read([1,2,3],window=win,masked=True)
        valid=ds.dataset_mask(window=win)>0
        valid &= ~np.ma.getmaskarray(a).any(axis=0)
        x=np.asarray(a.filled(0),dtype=np.float32)
        # RGB products expected standard 8-bit; prevent silent incorrect scaling.
        if np.issubdtype(np.dtype(ds.dtypes[0]),np.integer) and np.dtype(ds.dtypes[0]).itemsize==1:
            x=x/255.
        else:
            raise RuntimeError(f'Non-uint8 RGB {ds.dtypes}: {raster_path}; define radiometric scaling first')
    return x,np.asarray(valid,bool)
def supervision(mosaic,win,fold,role):
    site=RGB[mosaic]['site'];t=RGB[mosaic]['transform']
    h,w=int(win.height),int(win.width)
    affine=rasterio.windows.transform(win,t)
    footprint=box(*rasterio.windows.bounds(win,t))
    def burn(gs):
        return rasterize(((g,1) for g in gs if g is not None and not g.is_empty),
            out_shape=(h,w),transform=affine,fill=0,dtype='uint8').astype(bool) if gs else np.zeros((h,w),bool)
    roi=burn(geom_hits(ROI_T,ROI_GEOMS,footprint))
    pids=[int(i) for i in P_T.query(footprint,predicate='intersects')
          if PLOTS.iloc[int(i)].site_key==site and
          (role=='train' and PLOTS.iloc[int(i)].cv_fold!=fold or
           role=='val' and PLOTS.iloc[int(i)].cv_fold==fold)]
    plot=burn([P_GEOMS[i] for i in pids])
    # Every other fold's field plot is out of scope for val; all exhaustive
    # plot geometries (not only selected) override circle labels.
    allplot=burn([P_GEOMS[int(i)] for i in P_T.query(footprint,predicate='intersects')
                  if PLOTS.iloc[int(i)].site_key==site])
    cids=[int(i) for i in C_T.query(footprint,predicate='intersects')
          if CIRCLE_ROI.iloc[int(i)].site_key==site and
          not CIRCLE_ROI.iloc[int(i)].cross_fold_plot_intersection and
          (role=='train' and CIRCLE_ROI.iloc[int(i)].cv_fold!=fold)]
    circle=burn([C_GEOMS[i] for i in cids])&~allplot if role=='train' else np.zeros((h,w),bool)
    posplot=burn(geom_hits(CP_T,CROWN_POS,footprint))&plot
    negplot=burn(geom_hits(CN_T,CROWN_NEG,footprint))&plot
    poscircle=burn(geom_hits(KPOS_T,CIRCLE_POS_GEOMS,footprint))&circle
    part_pos=np.zeros((h,w),bool);part_neg=np.zeros((h,w),bool)
    if role=='train':
        positives,negatives,tp,tn=PARTIAL_T[site]
        part_pos=burn(geom_hits(tp,positives,footprint))&~allplot&~circle
        part_neg=burn(geom_hits(tn,negatives,footprint))&~allplot&~circle
    target=(posplot|poscircle|part_pos)
    conflicting=(posplot&negplot)|(part_pos&part_neg)
    weight=roi&(plot|circle|part_pos|part_neg)&~conflicting
    return target.astype(np.float32),weight.astype(np.float32),plot.astype(bool)
def get_chip(mosaic,win,fold,role):
    x,rgbvalid=rgb_window(RGB[mosaic]['path'],win)
    y,w,plot=supervision(mosaic,win,fold,role)
    w*=rgbvalid.astype(np.float32)
    return x,y,w,plot
print('Prepared lazy RGB + spatially strict masks. No imagery loaded globally.')


## 02 — V3_1 chip preparation and resumable cache (once per spatial fold)

As in V3_1, select a balanced subset of **physical windows**, group by mosaic, open each image once, extract **256 × 256 RGB + label + loss-mask chips**, and write compressed uint8 NPZ chunks. On rerun the fingerprint permits a **cache hit**. During GPU training, the dataset reads only RAM arrays—not Google Drive rasters. Small smoke settings cap the number of source windows per site.


In [ ]:
# 03 — V3_1-style resumable cache, file provenance, persistent raster handles
from contextlib import contextmanager
import hashlib, shutil
from collections import defaultdict
from rasterio.windows import Window
CACHE_DIR=OUT/'chip_cache';CACHE_DIR.mkdir(parents=True,exist_ok=True)
USE_LOCAL_STAGE=False  # Same optional one-mosaic SSD staging as V3_1; disabled by default
STAGE_DIR=Path('/content/musanga_v4_stage');STAGE_DIR.mkdir(parents=True,exist_ok=True)
@contextmanager
def staged_mosaic(name):
    global _ACTIVE_RASTER,_ACTIVE_NAME
    src=Path(RGB[name]['path']);selected=src
    local=STAGE_DIR/src.name
    if USE_LOCAL_STAGE and shutil.disk_usage('/content').free>src.stat().st_size+2*1024**3:
        if not local.exists() or local.stat().st_size!=src.stat().st_size:
            tmp=local.with_name(local.name+'.part')
            with src.open('rb') as a,tmp.open('wb') as b:
                shutil.copyfileobj(a,b,16*1024*1024)
            tmp.replace(local)
        selected=local
    assert _ACTIVE_RASTER is None
    try:
        with rasterio.open(selected) as ds:
            assert ds.transform==RGB[name]['transform'] and ds.count>=3
            _ACTIVE_RASTER=ds;_ACTIVE_NAME=name
            yield
    finally:
        _ACTIVE_RASTER=None;_ACTIVE_NAME=None
        if selected==local:local.unlink(missing_ok=True)
def cache_signature(fold,role):
    dependencies=[Path(CONTRACT['cv_manifest']),Path(CONTRACT['input_manifest']),
        Path(NB00['exhaustive_plot_crowns']),Path(NB00['plot_file']),
        Path(NB00['circle_positive_crowns']),Path(NB00['circle_complete_roi'])]
    dependencies += [Path(p) for p in NB00['partial_site_crowns'].values()]
    source_state=[(str(p),p.stat().st_size,p.stat().st_mtime_ns) for p in dependencies]
    info=dict(version='v4_resnet18_v3cache_v1',fold=int(fold),role=role,
        seed=SEED,chip=CHIP,smoke=SMOKE_TEST,src=source_state,
        max_train=MAX_TRAIN_WINDOWS_PER_SITE,max_val=MAX_VAL_WINDOWS_PER_SITE)
    return hashlib.sha256(json.dumps(info,sort_keys=True).encode()).hexdigest()
def pack(path,items,signature):
    tmp=path.with_name(path.name+'.tmp.npz')
    np.savez_compressed(tmp,key=np.array(signature),
        rgb=np.stack([r['rgb'] for r in items]).astype(np.uint8),
        y=np.stack([r['y'] for r in items]).astype(np.uint8),
        w=np.stack([r['w'] for r in items]).astype(np.uint8),
        site=np.array([r['site'] for r in items],dtype='U20'),
        mosaic=np.array([r['mosaic'] for r in items],dtype='U100'),
        row=np.array([r['row'] for r in items],dtype=np.int32),
        col=np.array([r['col'] for r in items],dtype=np.int32))
    tmp.replace(path)
def unpack(path,signature):
    with np.load(path,allow_pickle=False) as z:
        assert str(z['key'].item())==signature
        return [dict(rgb=z['rgb'][i],y=z['y'][i],w=z['w'][i],
            site=str(z['site'][i]),mosaic=str(z['mosaic'][i]),
            row=int(z['row'][i]),col=int(z['col'][i])) for i in range(len(z['rgb']))]
def source_window_rows(fold,role):
    q=SPLITS[(SPLITS.cv_fold==fold)&(SPLITS.split==role)].copy()
    assert len(q)>0,(fold,role)
    cap=MAX_TRAIN_WINDOWS_PER_SITE if role=='train' else MAX_VAL_WINDOWS_PER_SITE
    q['likely_positive']=q.apply(lambda r:bool(
        CP_T.query(box(*rasterio.windows.bounds(
            Window(int(r.col),int(r.row),int(r.width),int(r.height)),
            RGB[r.mosaic]['transform'])),predicate='intersects').size),axis=1)
    selected=[]
    for site,g in q.groupby('site',sort=True):
        # Approximate stratification by positive intersection before any image reads.
        high=g[g.likely_positive];low=g[~g.likely_positive]
        if role=='train' and len(high) and len(low):
            nh=min(len(high),int(cap*.65))
            chosen=pd.concat([high.sample(n=nh,random_state=SEED+fold),
                 low.sample(n=min(len(low),cap-nh),random_state=SEED+fold)])
        else:
            chosen=g.sample(n=min(cap,len(g)),random_state=SEED+fold)
        selected.append(chosen)
    return pd.concat(selected,ignore_index=True)
def make_cache(fold,role):
    sig=cache_signature(fold,role)
    cache=CACHE_DIR/f'field78_{role}_fold{fold}_{sig[:16]}.npz'
    if cache.exists():
        try:
            items=unpack(cache,sig)
            print('CACHE HIT',role,'fold',fold,'n',len(items),flush=True)
            return items
        except Exception as exc:
            print('Invalid cache:',exc)
            cache.unlink(missing_ok=True)
    selected=source_window_rows(fold,role)
    print('PREPARE',role,'fold',fold,
        selected.groupby('site').size().to_dict(),flush=True)
    parts_dir=CACHE_DIR/f'{role}_fold{fold}_{sig[:16]}'
    parts_dir.mkdir(exist_ok=True)
    items=[]
    t0=time.perf_counter()
    for mosaic,g in selected.groupby('mosaic',sort=True):
        with staged_mosaic(mosaic):
            for k in range(0,len(g),50):
                part=parts_dir/f'{mosaic}_{k:04d}.npz'
                if part.exists():
                    try:
                        existing=unpack(part,sig)
                        items.extend(existing);continue
                    except Exception:part.unlink(missing_ok=True)
                batch=[]
                for r in list(g.iloc[k:k+50].itertuples(index=False)):
                    # Same deterministic centre crop as V3_1; do not read 512px
                    # merely to throw away 3/4 of the source pixel data.
                    if r.width<CHIP or r.height<CHIP:continue
                    row=int(r.row)+(int(r.height)-CHIP)//2
                    col=int(r.col)+(int(r.width)-CHIP)//2
                    win=Window(col,row,CHIP,CHIP)
                    x,y,w,_=get_chip(mosaic,win,fold,role)
                    if w.sum()<50:continue
                    batch.append(dict(rgb=np.rint(np.clip(x.transpose(1,2,0),0,1)*255).astype('uint8'),
                        y=y.astype('uint8'),w=w.astype('uint8'),
                        site=RGB[mosaic]['site'],mosaic=mosaic,row=row,col=col))
                if batch:
                    pack(part,batch,sig)
                    items.extend(batch)
                print('Cached',role,'fold',fold,mosaic,
                    min(k+50,len(g)),'/',len(g),'usable',len(items),
                    'elapsed_s',round(time.perf_counter()-t0,1),flush=True)
    assert len(items)>0,(fold,role,'No valid RGB+reference chips')
    pack(cache,items,sig)
    print('CACHE SAVED',role,'fold',fold,'chips',len(items),
        'seconds',round(time.perf_counter()-t0,1),flush=True)
    return items


In [ ]:
# 04 — V3_1-style RAM-backed dataset (no GDAL calls inside GPU training)
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
class CachedChips(Dataset):
    def __init__(self,items,augment=False):
        self.items=items;self.augment=augment
    def __len__(self):return len(self.items)
    def __getitem__(self,i):
        q=self.items[i]
        rgb=q['rgb'];y=q['y'];w=q['w']
        if self.augment:
            k=int(torch.randint(0,4,(1,)).item())
            rgb=np.rot90(rgb,k);y=np.rot90(y,k);w=np.rot90(w,k)
            if torch.rand(1).item()<.5:
                rgb=rgb[:,::-1];y=y[:,::-1];w=w[:,::-1]
        x=torch.from_numpy(np.ascontiguousarray(rgb.transpose(2,0,1))).float()/255.
        return x,torch.from_numpy(np.ascontiguousarray(y[None])).float(),torch.from_numpy(np.ascontiguousarray(w[None])).float(),q['site']
def cached_sampler(items):
    counts=pd.Series([r['site'] for r in items]).value_counts().to_dict()
    weights=[(POSITIVE_SAMPLE_BOOST if np.any(r['y']&r['w']) else 1.)/counts[r['site']]
             for r in items]
    return WeightedRandomSampler(torch.as_tensor(weights,dtype=torch.double),
        num_samples=min(len(items),STEPS_PER_EPOCH*BATCH_SIZE),replacement=True)
print('Training uses 100% RAM-backed V3_1-style cached chips.')


## 03 — Train the original V3_1 pretrained ResNet18 U-Net, using mixed precision

**Same backbone as V3_1:** `segmentation_models_pytorch.Unet(encoder_name='resnet18', encoder_weights='imagenet', in_channels=3, classes=1)`. Masks and RGB chips are loaded from the resumable NPZ cache once. Augmentations are applied in RAM. Mixed precision (CUDA AMP) speeds up GPU training. Validation is site-macro averaged. The initial smoke run uses **one epoch**; this is an engineering check only.


In [ ]:
# 05 — Restore the working V3_1 ResNet18 U-Net + masked BCE, AMP in training
import segmentation_models_pytorch as smp
import torch.nn.functional as F
class ResNet18UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=smp.Unet(encoder_name='resnet18',encoder_weights='imagenet',
            in_channels=3,classes=1)
    def forward(self,x):return self.net(x)  # B,1,H,W
def masked_loss(logits,y,weight):
    bce=F.binary_cross_entropy_with_logits(logits.float(),y.float(),reduction='none')
    return (bce*weight).sum()/weight.sum().clamp_min(1.)
AMP=DEVICE.type=='cuda'
def run_epoch(model,loader,optimizer=None,scaler=None):
    model.train(optimizer is not None)
    totals=[];site_losses={}
    for x,y,w,site in loader:
        x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
        w=w.to(DEVICE,non_blocking=True)
        if not w.any():continue
        with torch.set_grad_enabled(optimizer is not None):
            with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=AMP):
                logits=model(x)
            loss=masked_loss(logits,y,w)
            if optimizer is not None:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer);scaler.update()
        totals.append(loss.detach().item())
        if optimizer is None:
            # Per-chip, equal-site loss (avoid ForestGEO pixel-volume dominance)
            for i,name in enumerate(site):
                if w[i].sum()==0:continue
                l=masked_loss(logits[i:i+1],y[i:i+1],w[i:i+1])
                site_losses.setdefault(str(name),[]).append(float(l.item()))
    if optimizer is None:
        return float(np.mean([np.mean(v) for v in site_losses.values()])) if site_losses else float('inf')
    return float(np.mean(totals)) if totals else float('inf')


In [ ]:
# 06 — Prepare each fold once, then reuse precomputed chips for every epoch
HISTORY=[];CHECKPOINTS={}
for fold in RUN_FOLDS:
    assert fold in (0,1,2)
    t0=time.perf_counter()
    train_chips=make_cache(fold,'train')
    val_chips=make_cache(fold,'val')
    print('Total cache loading/preparation seconds:',round(time.perf_counter()-t0,1))
    trainloader=DataLoader(CachedChips(train_chips,augment=True),
        batch_size=BATCH_SIZE,sampler=cached_sampler(train_chips),
        num_workers=0,pin_memory=AMP)
    valloader=DataLoader(CachedChips(val_chips,augment=False),
        batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=AMP)
    model=ResNet18UNet().to(DEVICE)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=1e-4)
    scaler=torch.amp.GradScaler('cuda',enabled=AMP)
    best=float('inf');wait=0
    ckpt=OUT/f'V4_RGB_ResNet18_fold{fold}_best.pt'
    for epoch in range(1,EPOCHS+1):
        tic=time.perf_counter()
        trloss=run_epoch(model,trainloader,optimizer,scaler)
        valloss=run_epoch(model,valloader,None,None)
        item=dict(fold=fold,epoch=epoch,train_loss=trloss,
            val_loss_site_macro=valloss,epoch_seconds=round(time.perf_counter()-tic,1),
            n_train_chips=len(train_chips),n_val_chips=len(val_chips))
        HISTORY.append(item);print(item,flush=True)
        if valloss<best-1e-4:
            best=valloss;wait=0
            torch.save(dict(model=model.state_dict(),fold=fold,
                architecture='smp_unet_resnet18_imagenet',smoke=SMOKE_TEST,
                epoch=epoch,val_loss=best,cache_recipe='V3_1_resumable_npz'),ckpt)
        else:
            wait+=1
            if wait>=PATIENCE:break
    CHECKPOINTS[fold]=ckpt
    del train_chips,val_chips,trainloader,valloader,model
    if AMP:torch.cuda.empty_cache()
pd.DataFrame(HISTORY).to_csv(OUT/'02_rgb_training_history.csv',index=False)
print('Best V4 checkpoints:',CHECKPOINTS)


## 04 — Held-out accuracy by **individual plot** (not pooled pixels)

Evaluate the best checkpoint at a fixed preliminary threshold of 0.5. Each prediction contributes to a **single held-out plot**, with a special mask for that plot; only valid RGB and ROI pixels are counted. Report per-plot TP/FP/FN/TN, cover bias, MAE, IoU, Dice/F1, precision and recall. A zero-positive plot has undefined recall/F1, and is scored for false positives and cover error. Summaries use equal plot weights, and separately equal site weights. Threshold optimization, if eventually used, must be **nested within training data**, not fitted on the held-out plots.


In [ ]:
# 07 — Plot-wise confusion, fold-/site-balanced summaries and output provenance
def burn_single(g,win,affine):
    return rasterize([(g,1)],out_shape=(int(win.height),int(win.width)),
        transform=affine,fill=0,dtype='uint8').astype(bool)
PLOT_ROWS=[]
for fold in RUN_FOLDS:
    ck=torch.load(CHECKPOINTS[fold],map_location=DEVICE,weights_only=False)
    model=ResNet18UNet().to(DEVICE);model.load_state_dict(ck['model']);model.eval()
    fval=SPLITS[(SPLITS.cv_fold==fold)&(SPLITS.split=='val')].copy()
    if SMOKE_TEST:
        fval=fval.groupby('site',group_keys=False).head(MAX_VAL_WINDOWS_PER_SITE)
    plot_state={}
    psite=PLOTS[PLOTS.cv_fold==fold]
    for r in psite.itertuples(index=False):
        plot_state[str(r.Plot_name)]=dict(tp=0,fp=0,fn=0,tn=0,n=0,
                                          site=r.site_key,ba=float(r.pct_BA_musanga))
    with torch.no_grad():
        for row in fval.itertuples(index=False):
            # Disjoint tiles from 512px windows; no duplicate physical pixels.
            for dr in range(0,int(row.height),CHIP):
                for dc in range(0,int(row.width),CHIP):
                    hh=min(CHIP,int(row.height)-dr);ww=min(CHIP,int(row.width)-dc)
                    if hh!=CHIP or ww!=CHIP:continue
                    win=Window(int(row.col)+dc,int(row.row)+dr,CHIP,CHIP)
                    _,weights,_=supervision(row.mosaic,win,fold,'val')
                    x,valid=rgb_window(RGB[row.mosaic]['path'],win)
                    if not np.any(valid&weights.astype(bool)):continue
                    prediction=torch.sigmoid(model(torch.from_numpy(x[None]).to(DEVICE)))[0,0]
                    pred=prediction.cpu().numpy()>=.5
                    # Compute reference once. Plot identity comes from *actual*
                    # field geometry, not from a candidate window's source_id.
                    truth,base_weight,_=supervision(row.mosaic,win,fold,'val')
                    affine=rasterio.windows.transform(win,RGB[row.mosaic]['transform'])
                    ext=box(*rasterio.windows.bounds(win,RGB[row.mosaic]['transform']))
                    candidates=P_T.query(ext,predicate='intersects')
                    for i in candidates:
                        p=PLOTS.iloc[int(i)]
                        if p.cv_fold!=fold or p.site_key!=row.site:continue
                        plotmask=burn_single(p.geometry,win,affine)
                        mask=plotmask&valid&base_weight.astype(bool)
                        if not mask.any():continue
                        obs=truth.astype(bool)
                        entry=plot_state[str(p.Plot_name)]
                        entry['tp']+=int((mask&pred&obs).sum())
                        entry['fp']+=int((mask&pred&~obs).sum())
                        entry['fn']+=int((mask&~pred&obs).sum())
                        entry['tn']+=int((mask&~pred&~obs).sum())
                        entry['n']+=int(mask.sum())
    for name,d in plot_state.items():
        tp,fp,fn,tn=d['tp'],d['fp'],d['fn'],d['tn']
        n=d['n'];pos=tp+fn;predpos=tp+fp
        PLOT_ROWS.append(dict(fold=fold,Plot_name=name,site=d['site'],
             n_rgb_valid_label_pixels=n,field_BA_pct=d['ba'],
             tp=tp,fp=fp,fn=fn,tn=tn,
             crown_reference_pct=100*pos/n if n else np.nan,
             predicted_cover_pct=100*predpos/n if n else np.nan,
             cover_bias_pp=100*(predpos-pos)/n if n else np.nan,
             cover_abs_error_pp=100*abs(predpos-pos)/n if n else np.nan,
             pixel_precision=tp/(tp+fp) if tp+fp else np.nan,
             pixel_recall=tp/pos if pos else np.nan,
             pixel_F1=2*tp/(2*tp+fp+fn) if pos and (2*tp+fp+fn)>0 else np.nan,
             pixel_IoU=tp/(tp+fp+fn) if pos and tp+fp+fn>0 else np.nan,
             false_positive_fraction_pct=100*fp/n if n else np.nan))
RESULTS=pd.DataFrame(PLOT_ROWS)
RESULTS.to_csv(OUT/'02_rgb_heldout_per_plot.csv',index=False)
SUMMARY=RESULTS.groupby(['fold','site'],as_index=False).agg(
    n_plots=('Plot_name','size'),
    plots_with_evaluation=('n_rgb_valid_label_pixels',lambda x:int((x>0).sum())),
    mean_cover_MAE_pp=('cover_abs_error_pp','mean'),
    mean_pixel_IoU=('pixel_IoU','mean'),
    mean_pixel_F1=('pixel_F1','mean'),
    mean_false_positive_fraction_pct=('false_positive_fraction_pct','mean'))
SUMMARY.to_csv(OUT/'02_rgb_heldout_by_fold_site.csv',index=False)
display(SUMMARY.round(3))
print('Equal-site mean plot MAE (pp):',
      round(float(SUMMARY.mean_cover_MAE_pp.mean()),3))
if SMOKE_TEST:
    print('SMOKE TEST ONLY: fold 0, partial validation windows; do not use as scientific accuracy.')
else:
    assert set(RESULTS.fold)=={0,1,2}
    print('Full 3-fold spatial CV results saved.')
RUN_INFO=dict(schema='musanga_v4_rgb_unet_baseline_1',
    smoke_test=SMOKE_TEST,folds=RUN_FOLDS,epochs=EPOCHS,
    patches_per_epoch=STEPS_PER_EPOCH*BATCH_SIZE,chip=CHIP,
    final_manifest=CONTRACT['cv_manifest'],threshold=.5,
    macro_rule='per plot, then per site; zero-positive F1 undefined',
    checkpoints={str(f):str(p) for f,p in CHECKPOINTS.items()},
    model_status='trained_smoke_only' if SMOKE_TEST else 'trained_full_spatial_cv')
(OUT/'02_run_contract.json').write_text(json.dumps(RUN_INFO,indent=2))
